In [12]:
import os
import time

import cv2
import pandas as pd
import torch
from natsort import natsorted
from ultralytics import YOLO

from code_programm.path import get_path_weight_model

In [13]:
print(torch.cuda.device_count())
print(torch.cuda.get_device_name())
model_speed = YOLO(get_path_weight_model('speed_recognition.pt'))

1
NVIDIA GeForce GTX 1080 Ti


In [14]:
wheel_ets_train_speed = pd.DataFrame(columns=['speed'])

In [11]:
paths = [
    r'D:\Dataset_for_autopilot\2024-04-08 14-25-52',
    r'D:\Dataset_for_autopilot\2024-04-08 14-25-43',
    r'D:\Dataset_for_autopilot\2024-04-08 14-25-34',
    r'D:\Dataset_for_autopilot\2024-04-08 14-25-26',
    r'D:\Dataset_for_autopilot\2024-04-08 14-25-07',
    r'D:\Dataset_for_autopilot\2024-04-08 14-24-51',
    r'D:\Dataset_for_autopilot\2024-04-08 14-23-11',
    r'D:\Dataset_for_autopilot\2024-04-08 14-20-50',
    r'D:\Dataset_for_autopilot\2024-04-08 14-19-43',
    r'D:\Dataset_for_autopilot\2024-04-08 14-18-55',
    r'D:\Dataset_for_autopilot\2024-04-08 14-17-38',
    r'D:\Dataset_for_autopilot\2024-04-08 14-15-59',
    r'D:\Dataset_for_autopilot\2024-04-08 14-15-30',
    r'D:\Dataset_for_autopilot\2024-04-08 14-11-02',
]

In [15]:
paths = [
    r'D:\Dataset_for_autopilot\2024-04-08 14-00-05',
]

In [16]:
for j in paths:
    path_i = os.path.join(j, f'speed')
    if os.path.exists(f'{path_i}') and os.path.isdir(f'{path_i}'):
        speed_files = [os.path.join(path_i, file) for file in os.listdir(path_i) if file.endswith('.png')]
        print("Полные пути к файлам в папке:")
        speed_files = natsorted(speed_files)
        print(speed_files[0])
    else:
        print("Указанный путь не существует или не является папкой.")
    length = len(wheel_ets_train_speed)
    start_time = time.time()
    for file in speed_files:
        bgra_image = cv2.imread(file, cv2.IMREAD_UNCHANGED)
        scaled_image = cv2.cvtColor(bgra_image, cv2.COLOR_BGRA2BGR)
        scaled_image = cv2.resize(scaled_image, None, fx=2, fy=2, interpolation=cv2.INTER_LINEAR)
        results = model_speed.predict(scaled_image, conf=0.8, device='cuda', verbose=False, show=False)
        sorted_objects = sorted(
            ({'class': int(cls), 'confidence': float(conf), 'xmin': int(xmin), 'ymin': int(ymin), 'xmax': int(xmax),
              'ymax': int(ymax)}
             for result in results for obj in result.boxes.data for xmin, ymin, xmax, ymax, conf, cls in
             (obj.tolist(),)),
            key=lambda obj: obj['xmin']
        )
        if sorted_objects:
            speed = ''.join(str(obj['class']) for obj in sorted_objects)
            speed = int(speed)
        else:
            speed = wheel_ets_train_speed.iloc[len(wheel_ets_train_speed) - 1]
        wheel_ets_train_speed.loc[len(wheel_ets_train_speed)] = speed
    
    wheel_ets_train_speed = wheel_ets_train_speed.drop(wheel_ets_train_speed.tail(1).index, axis=0)
    
    print('Изображений:', len(wheel_ets_train_speed) - length, '\nСек:', time.time() - start_time, '\nCек\изображение:',
          (time.time() - start_time) / (len(wheel_ets_train_speed) - length), '\n')
    
print('Всего:', len(wheel_ets_train_speed))

Полные пути к файлам в папке:
D:\Dataset_for_autopilot\2024-04-08 14-00-05\speed\2024-04-08 14-00-05_0.png
Изображений: 17050 
Сек: 184.78884601593018 
Cек\изображение: 0.010838055484805289 

Всего: 17050


In [17]:
len(wheel_ets_train_speed)

17050

In [18]:
wheel_ets_train_speed.to_csv(r'C:\PycharmProjects\ETS_Autopilot\dataset_for_wheel_nn\version_4\X_train_speed_4_2.csv', index=False)

In [42]:
wheel_ets_train_speed_ = pd.read_csv(r'C:\PycharmProjects\ETS_Autopilot\dataset_for_wheel_nn\version_4\X_train_road_4_1.csv')
len(wheel_ets_train_speed_)

13587

In [8]:
wheel_ets_train_speed_.loc[2]

speed    58
Name: 2, dtype: int64

In [9]:
path_i = os.path.join(paths[3], f'speed')
if os.path.exists(f'{path_i}') and os.path.isdir(f'{path_i}'):
    png_files = [os.path.join(path_i, file) for file in os.listdir(path_i) if file.endswith('.png')]
    png_files = natsorted(png_files)

In [10]:
bgra_image = cv2.imread(png_files[400], cv2.IMREAD_UNCHANGED)
scaled_image = cv2.cvtColor(bgra_image, cv2.COLOR_BGRA2BGR)
cv2.imshow('BGR Image', bgra_image)
cv2.waitKey(0)
cv2.destroyAllWindows()
# scaled_image = cv2.resize(scaled_image, None, fx=2, fy=2, interpolation=cv2.INTER_LINEAR)
results = model_speed.predict(scaled_image,
                              conf=0.8,
                              device='cuda',
                              verbose=False,
                              show=True
                              )
cv2.waitKey(0)
cv2.destroyAllWindows()
sorted_objects = sorted(
    ({'class': int(cls), 'confidence': float(conf), 'xmin': int(xmin), 'ymin': int(ymin), 'xmax': int(xmax),
      'ymax': int(ymax)}
     for result in results for obj in result.boxes.data for xmin, ymin, xmax, ymax, conf, cls in
     (obj.tolist(),)),
    key=lambda obj: obj['xmin']
)
if sorted_objects:
    speed = ''.join(str(obj['class']) for obj in sorted_objects)
else:
    speed = wheel_ets_train_speed.iloc[len(wheel_ets_train_speed) - 1]
print(speed)

119


In [11]:
cv2.destroyAllWindows()